## Depth TargetLayer 实例化调试

这个 notebook 用 `data/raw/mero_84_coord_extend` 的 SEG-Y 几何测试深度解释层位能否建立 `TargetLayer`。这份 SEG-Y 的采样头虽然显示为时间/毫秒，但这里按深度米处理：4750-7500 m，采样间隔 5 m。

层位顺序按浅到深：`base_of_salt_extend`、`base_of_bve_extend`、`base_of_itp_extend`。运行后会先做全量相邻层位顺序预检查，再尝试实例化 `TargetLayer`；如果出现底高于顶或顶底相等，会输出违例点 CSV 和图。 


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_interpretation_petrel
from cup.seismic.process import TargetLayer, interpolate_interpretation_surface
from cup.seismic.survey import open_survey

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True


In [ ]:
data_root = repo_root / "data"
debug_dir = data_root / "_debug" / "target_layer_depth_instantiation@20260422"
debug_dir.mkdir(parents=True, exist_ok=True)

seismic_file = data_root / "raw" / "mero_84_coord_extend"
segy_iline = 5
segy_xline = 21
segy_istep = 1
segy_xstep = 4

horizon_files = {
    "base_of_salt_extend": data_root / "interpre_depth" / "base_of_salt_extend",
    "base_of_bve_extend": data_root / "interpre_depth" / "base_of_bve_extend",
    "base_of_itp_extend": data_root / "interpre_depth" / "base_of_itp_extend",
}

# Depth-domain isolated outlier threshold, in meters. Adjust this if QC shows too many removed points.
outlier_threshold_m = 50.0

for path in [seismic_file, *horizon_files.values()]:
    if not path.exists():
        raise FileNotFoundError(path)

debug_dir


In [ ]:
def _finite_min_max(values: pd.Series) -> tuple[float, float]:
    arr = values.to_numpy(dtype=float, copy=False)
    finite = np.isfinite(arr)
    if not np.any(finite):
        return float("nan"), float("nan")
    return float(np.nanmin(arr[finite])), float(np.nanmax(arr[finite]))


def build_interpolated_horizons(geometry: dict) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    interpolated = {}
    qc_records = []

    for horizon_name, horizon_file in horizon_files.items():
        raw_df = import_interpretation_petrel(horizon_file)
        raw_min, raw_max = _finite_min_max(raw_df["interpretation"])

        interp_df = interpolate_interpretation_surface(
            interpretation_df=raw_df,
            geometry=geometry,
            outlier_threshold=outlier_threshold_m,
            min_neighbor_count=2,
            keep_nan=True,
        )
        interp_min, interp_max = _finite_min_max(interp_df["interpretation"])
        outlier_stats = dict(interp_df.attrs.get("outlier_removal", {}))

        interpolated[horizon_name] = interp_df
        qc_records.append(
            {
                "horizon": horizon_name,
                "file": horizon_file.as_posix(),
                "raw_rows": len(raw_df),
                "raw_depth_min": raw_min,
                "raw_depth_max": raw_max,
                "interpolated_rows": len(interp_df),
                "interpolated_valid_rows": int(np.isfinite(interp_df["interpretation"]).sum()),
                "interpolated_depth_min": interp_min,
                "interpolated_depth_max": interp_max,
                "removed_points": outlier_stats.get("removed_points"),
                "removed_ratio": outlier_stats.get("removed_ratio"),
                "outlier_threshold_m": outlier_stats.get("threshold"),
            }
        )

    return interpolated, pd.DataFrame.from_records(qc_records)


def collect_pair_order_violations(interpolated: dict[str, pd.DataFrame]) -> pd.DataFrame:
    records = []
    horizon_names = list(horizon_files.keys())

    for top_name, bottom_name in zip(horizon_names[:-1], horizon_names[1:]):
        top_df = interpolated[top_name][["inline", "xline", "interpretation"]].rename(
            columns={"interpretation": "interpretation_top"}
        )
        bottom_df = interpolated[bottom_name][["inline", "xline", "interpretation"]].rename(
            columns={"interpretation": "interpretation_bottom"}
        )
        aligned = pd.merge(top_df, bottom_df, on=["inline", "xline"], how="inner")
        finite = np.isfinite(aligned["interpretation_top"]) & np.isfinite(aligned["interpretation_bottom"])
        aligned = aligned.loc[finite].copy()
        aligned["interpretation_delta"] = aligned["interpretation_top"] - aligned["interpretation_bottom"]

        violated = aligned.loc[aligned["interpretation_delta"] >= 0.0].copy()
        if violated.empty:
            continue

        violated.insert(0, "bottom_name", bottom_name)
        violated.insert(0, "top_name", top_name)
        records.append(violated)

    if not records:
        return pd.DataFrame(
            columns=[
                "top_name",
                "bottom_name",
                "inline",
                "xline",
                "interpretation_top",
                "interpretation_bottom",
                "interpretation_delta",
            ]
        )

    return pd.concat(records, ignore_index=True).sort_values(
        ["top_name", "bottom_name", "inline", "xline"]
    ).reset_index(drop=True)


def summarize_pair_order_violations(violation_df: pd.DataFrame) -> pd.DataFrame:
    if violation_df.empty:
        return pd.DataFrame()
    return violation_df.groupby(["top_name", "bottom_name"], as_index=False).agg(
        point_count=("interpretation_delta", "size"),
        delta_min=("interpretation_delta", "min"),
        delta_max=("interpretation_delta", "max"),
        top_depth_min=("interpretation_top", "min"),
        top_depth_max=("interpretation_top", "max"),
        bottom_depth_min=("interpretation_bottom", "min"),
        bottom_depth_max=("interpretation_bottom", "max"),
        inline_min=("inline", "min"),
        inline_max=("inline", "max"),
        xline_min=("xline", "min"),
        xline_max=("xline", "max"),
    )


def instantiate_target_layer(geometry: dict):
    existing_logs = set(debug_dir.glob("horizon_order_violation__*.csv"))
    interpolated, horizon_qc_df = build_interpolated_horizons(geometry)
    pair_violation_df = collect_pair_order_violations(interpolated)

    pair_violation_path = None
    if not pair_violation_df.empty:
        pair_violation_path = debug_dir / "all_pair_order_violations_precheck.csv"
        pair_violation_df.to_csv(pair_violation_path, index=False, encoding="utf-8-sig")

    try:
        target_layer = TargetLayer(
            interpolated_horizon_dfs=interpolated,
            geometry=geometry,
            horizon_names=list(horizon_files.keys()),
            debug_dir=debug_dir,
        )
        return target_layer, None, {
            "interpolated": interpolated,
            "horizon_qc_df": horizon_qc_df,
            "pair_violation_df": pair_violation_df,
            "pair_violation_path": pair_violation_path,
            "targetlayer_violation_df": None,
            "targetlayer_log_path": None,
        }
    except AssertionError as exc:
        all_logs = set(debug_dir.glob("horizon_order_violation__*.csv"))
        new_logs = sorted(all_logs - existing_logs, key=lambda path: path.stat().st_mtime, reverse=True)
        historical_logs = sorted(all_logs, key=lambda path: path.stat().st_mtime, reverse=True)
        log_path = new_logs[0] if new_logs else (historical_logs[0] if historical_logs else None)
        targetlayer_violation_df = pd.read_csv(log_path) if log_path is not None else None
        return None, exc, {
            "interpolated": interpolated,
            "horizon_qc_df": horizon_qc_df,
            "pair_violation_df": pair_violation_df,
            "pair_violation_path": pair_violation_path,
            "targetlayer_violation_df": targetlayer_violation_df,
            "targetlayer_log_path": log_path,
        }


In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": segy_iline,
        "xline": segy_xline,
        "istep": segy_istep,
        "xstep": segy_xstep,
    },
)
geometry = survey.query_geometry(domain="depth")

geometry_df = pd.DataFrame.from_dict(geometry, orient="index", columns=["value"])
display(geometry_df)

target_layer, target_layer_error, debug_payload = instantiate_target_layer(geometry)
horizon_qc_df = debug_payload["horizon_qc_df"]
pair_violation_df = debug_payload["pair_violation_df"]
pair_violation_path = debug_payload["pair_violation_path"]
targetlayer_violation_df = debug_payload["targetlayer_violation_df"]
targetlayer_log_path = debug_payload["targetlayer_log_path"]

print("Horizon interpolation QC:")
display(horizon_qc_df)

if pair_violation_df.empty:
    print("Precheck: no adjacent horizon order violations found.")
else:
    print("Precheck violation CSV:", pair_violation_path)
    print("Precheck violation count:", len(pair_violation_df))
    display(summarize_pair_order_violations(pair_violation_df))
    display(pair_violation_df.head(30))

if target_layer is not None:
    print("TargetLayer instantiated successfully.")
    print("Horizon names:", target_layer.horizon_names)
    print("Zones:", target_layer.iter_zones())
else:
    print("TargetLayer failed:")
    print(target_layer_error)
    print("TargetLayer debug log:", targetlayer_log_path)
    if targetlayer_violation_df is not None:
        print("TargetLayer first failing pair violation count:", len(targetlayer_violation_df))
        display(targetlayer_violation_df.head(30))


In [ ]:
plot_violation_df = pair_violation_df if not pair_violation_df.empty else targetlayer_violation_df

if plot_violation_df is None or plot_violation_df.empty:
    print("No violation points to visualize.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    scatter = axes[0].scatter(
        plot_violation_df["inline"],
        plot_violation_df["xline"],
        c=plot_violation_df["interpretation_delta"],
        cmap="coolwarm",
        s=18,
        alpha=0.75,
        edgecolors="none",
    )
    axes[0].set_title("Adjacent Horizon Order Violations")
    axes[0].set_xlabel("inline")
    axes[0].set_ylabel("xline")
    axes[0].set_xlim(float(geometry["inline_min"]), float(geometry["inline_max"]))
    axes[0].set_ylim(float(geometry["xline_min"]), float(geometry["xline_max"]))
    colorbar = fig.colorbar(scatter, ax=axes[0])
    colorbar.set_label("top - bottom (m)")

    axes[1].hist(plot_violation_df["interpretation_delta"].to_numpy(dtype=float), bins=50, color="#c44e52")
    axes[1].set_title("Violation Delta Histogram")
    axes[1].set_xlabel("top - bottom (m)")
    axes[1].set_ylabel("count")

    fig.suptitle(f"{len(plot_violation_df)} violation point(s)", fontsize=12)
    fig.tight_layout()


In [ ]:
if target_layer is None:
    print("TargetLayer is unavailable; fix horizon order violations before mask/sample-index checks.")
else:
    for zone in target_layer.iter_zones():
        top_grid, bottom_grid = target_layer.get_zone_sample_index_grids(zone)
        valid = np.isfinite(top_grid) & np.isfinite(bottom_grid)
        print(
            zone,
            "valid cells:", int(np.count_nonzero(valid)),
            "top sample index range:", (float(np.nanmin(top_grid)), float(np.nanmax(top_grid))),
            "bottom sample index range:", (float(np.nanmin(bottom_grid)), float(np.nanmax(bottom_grid))),
        )
